### Дообучение модели на данных из TMDB

In [40]:
import os
import sys
import re
import json

from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

from datasets import Dataset as DatasetForTransformers
from datasets import ClassLabel, Features, Value

load_dotenv()

True

Возьмем данные об отзывах с сайта TMDB из бд

In [35]:
sys.path.append('C:/Users/retro/ILYA/pet-projects/review-sentiment-analysis')

DATABASE_URL = os.getenv('DATABASE_URL') 

engine = create_engine(DATABASE_URL)

SessionLocal = sessionmaker(bind=engine)

db = SessionLocal()

In [42]:
from app.models import Review

reviews = db.query(Review).all()

review_texts = []
labels = []
for item in reviews:
    text = item.content.replace('<br />', ' ').strip()
    text = text.replace('\r', '').replace('\n', '').replace("\\'", "'")
    text = re.sub(r'<.*?>', '', text)

    if item.rating >= 7:
        labels.append(1)
        review_texts.append(text)
        
    elif item.rating <= 4:
        labels.append(0)
        review_texts.append(text)

In [43]:
with open('../data/test_data.json', 'r', encoding='utf-8') as file:
    test_data = json.load(file)

X_test = test_data['X']
y_test = test_data['y']

ds_train_val = DatasetForTransformers.from_dict({'text': review_texts, 'labels': labels})
ds_test = DatasetForTransformers.from_dict({'text': X_test, 'labels': y_test})

features = Features({
    'text': Value('string'),
    'labels': ClassLabel(names=['neg', 'pos'])
})

ds_train_val = ds_train_val.cast(features)
split = ds_train_val.train_test_split(test_size=0.1, seed=42, stratify_by_column='labels')
ds_train = split['train']
ds_val = split['test']

Casting the dataset: 100%|██████████| 97/97 [00:00<00:00, 12128.05 examples/s]


In [45]:
import numpy as np
import random
from datetime import datetime

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

import torch

from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

set_seed = seed

Дообучим модель с теми же параметрами обучения на новых данных полученных с TMDB

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

results_summary = []

model_path = "./best_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length'
        max_length=tokenizer.model_max_length
    )

train_tok = ds_train.map(tokenize_fn, batched=True, remove_columns=['text'])
val_tok = ds_val.map(tokenize_fn, batched=True, remove_columns=['text'])
test_tok = ds_test.map(tokenize_fn, batched=True, remove_columns=['text'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

args = TrainingArguments(
    output_dir=run_dir,
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,

    logging_steps=50,
    report_to='none',
    fp16=True,
    max_grad_norm=1.0,
    optim='adamw_torch'
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# --- VAL: classification_report ---
val_pred = trainer.predict(val_tok)
val_preds = np.argmax(val_pred.predictions, axis=1)
print('\nVAL classification_report:') 
print(classification_report(ds_val['labels'], val_preds, target_names=['neg', 'pos'], digits=4))

# --- TEST: classification_report --- 
test_pred = trainer.predict(test_tok) 
test_preds = np.argmax(test_pred.predictions, axis=1) 
print('\nTEST classification_report:') 
print(classification_report(ds_test['labels'], test_preds, target_names=['neg', 'pos'], digits=4))

report_dict = classification_report(
    ds_test['labels'],
    test_preds,
    target_names=['neg', 'pos'],
    output_dict=True
)

# Сохраним краткий итог
best_val = trainer.state.best_metric
results_summary.append({
    'model': model_name,
    'run_dir': run_dir,
    'best_val_f1': float(best_val) if best_val is not None else None,
    'test_accuracy': float(accuracy_score(ds_test['labels'], test_preds)),
    'test_report': report_dict
})

print('\n' + '='*80)
print('SUMMARY:')
for r in sorted(results_summary, key=lambda d: (d['best_val_f1'] is not None, d['best_val_f1']), reverse=True):
    print(r)

c:\Users\retro\ILYA\pet-projects\review-sentiment-analysis\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\retro\.cache\huggingface\hub\models--google--electra-base-discriminator. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 559.44it/s, Mat

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.635205,0.700000,0.700000,1.000000,0.823529
2,No log,0.613599,0.700000,0.700000,1.000000,0.823529
3,No log,0.607690,0.700000,0.700000,1.000000,0.823529


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]
There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.la


VAL classification_report:
              precision    recall  f1-score   support

         neg     0.0000    0.0000    0.0000         3
         pos     0.7000    1.0000    0.8235         7

    accuracy                         0.7000        10
   macro avg     0.3500    0.5000    0.4118        10
weighted avg     0.4900    0.7000    0.5765        10



c:\Users\retro\ILYA\pet-projects\review-sentiment-analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\retro\ILYA\pet-projects\review-sentiment-analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\retro\ILYA\pet-projects\review-sentiment-analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control thi


TEST classification_report:


c:\Users\retro\ILYA\pet-projects\review-sentiment-analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\retro\ILYA\pet-projects\review-sentiment-analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\retro\ILYA\pet-projects\review-sentiment-analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control thi

              precision    recall  f1-score   support

         neg     0.0000    0.0000    0.0000     12500
         pos     0.5000    1.0000    0.6667     12500

    accuracy                         0.5000     25000
   macro avg     0.2500    0.5000    0.3333     25000
weighted avg     0.2500    0.5000    0.3333     25000



c:\Users\retro\ILYA\pet-projects\review-sentiment-analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\retro\ILYA\pet-projects\review-sentiment-analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\retro\ILYA\pet-projects\review-sentiment-analysis\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control thi


SUMMARY:
{'model': 'google/electra-base-discriminator', 'run_dir': 'bert_runs/google_electra-base-discriminator_2026-03-13_18-00-18', 'best_val_f1': 0.8235294117647058, 'test_accuracy': 0.5, 'test_report': {'neg': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 12500.0}, 'pos': {'precision': 0.5, 'recall': 1.0, 'f1-score': 0.6666666666666666, 'support': 12500.0}, 'accuracy': 0.5, 'macro avg': {'precision': 0.25, 'recall': 0.5, 'f1-score': 0.3333333333333333, 'support': 25000.0}, 'weighted avg': {'precision': 0.25, 'recall': 0.5, 'f1-score': 0.33333333333333326, 'support': 25000.0}}}
